# 07.4 — Dense Passage Retrieval (DPR)

**DPR (Karpukhin et al., 2020):** Train two separate BERT encoders — one for questions, one for passages — so that relevant (question, passage) pairs have high dot product similarity.

**Why two encoders?** Questions and passages have different linguistic styles. A shared encoder works but two separate ones usually perform better (less cross-contamination).

**Key innovation over BM25:** DPR retrieves by meaning, not keyword overlap. "What is the capital of France?" retrieves "Paris is the seat of the French government" even with zero word overlap.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from dataclasses import dataclass
from typing import List

# ---- DPR Architecture ----
# Two encoders, independently applied.
# Both produce a single vector per input (from [CLS]).
# score(q, p) = E_q(q)^T * E_p(p)

class DPREncoder(nn.Module):
    """Lightweight stand-in for a full BERT encoder."""
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2, max_len=128):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            activation='gelu', batch_first=True, dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.d_model = d_model
    
    def forward(self, tokens):
        B, T = tokens.shape
        pos = torch.arange(T, device=tokens.device).unsqueeze(0)
        x = self.tok_emb(tokens) + self.pos_emb(pos)
        x = self.transformer(x)
        x = self.norm(x)
        return x[:, 0]  # [CLS] representation — [B, d_model]


class DPRModel(nn.Module):
    """
    DPR: two separate encoders for question and passage.
    In the original paper, both start from the same BERT checkpoint
    but are then fine-tuned independently.
    """
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2):
        super().__init__()
        self.question_encoder = DPREncoder(vocab_size, d_model, n_heads, n_layers)
        self.passage_encoder  = DPREncoder(vocab_size, d_model, n_heads, n_layers)
    
    def encode_question(self, tokens):
        return self.question_encoder(tokens)  # [B, d]
    
    def encode_passage(self, tokens):
        return self.passage_encoder(tokens)   # [B, d]
    
    def score(self, q_emb, p_emb):
        """Dot product similarity (no normalization in original DPR)."""
        return (q_emb * p_emb).sum(-1)  # [B]


VOCAB = 500
dpr = DPRModel(VOCAB)
n_params = sum(p.numel() for p in dpr.parameters())
print(f"DPR total params: {n_params:,}")
print(f"  Question encoder: {sum(p.numel() for p in dpr.question_encoder.parameters()):,}")
print(f"  Passage encoder:  {sum(p.numel() for p in dpr.passage_encoder.parameters()):,}")
print()
print("Real DPR: 2 × BERT-base (110M) = 220M parameters")

In [ ]:
# ---- DPR Training: In-batch Negatives ----
# Given a batch of (question, positive_passage) pairs:
# - question[i] should score high with passage[i]
# - question[i] should score low with passage[j] for j != i
# This creates B^2 - B negative pairs for free from a batch of B!

@dataclass
class DPRBatch:
    questions: torch.Tensor     # [B, T_q]
    pos_passages: torch.Tensor  # [B, T_p]  — the relevant passage
    neg_passages: torch.Tensor  # [B, T_p]  — an explicit hard negative (optional)


def dpr_loss(model: DPRModel, batch: DPRBatch, use_hard_negatives=True):
    """
    NLL loss over in-batch negatives.
    
    If hard negatives are included, passage matrix is [B, 2B] (pos + neg).
    Correct pair is still diagonal, but negatives include the hard ones.
    """
    q_emb = model.encode_question(batch.questions)      # [B, d]
    p_emb = model.encode_passage(batch.pos_passages)    # [B, d]
    
    if use_hard_negatives and batch.neg_passages is not None:
        n_emb = model.encode_passage(batch.neg_passages)  # [B, d]
        # Concatenate: [2B, d] — first B are positives, next B are negatives
        all_passages = torch.cat([p_emb, n_emb], dim=0)
    else:
        all_passages = p_emb
    
    # Similarity matrix: [B, n_passages]
    sim = q_emb @ all_passages.T  # dot product
    
    # Labels: question[i] matches passage[i] (index i in the positive block)
    labels = torch.arange(q_emb.size(0), device=q_emb.device)
    return F.cross_entropy(sim, labels)


# Training loop
optimizer = torch.optim.AdamW(dpr.parameters(), lr=2e-4)

def make_dpr_batch(B=8, T_q=10, T_p=20):
    return DPRBatch(
        questions=torch.randint(1, VOCAB, (B, T_q)),
        pos_passages=torch.randint(1, VOCAB, (B, T_p)),
        neg_passages=torch.randint(1, VOCAB, (B, T_p)),
    )

print("Training DPR with in-batch negatives + hard negatives:")
for step in range(200):
    batch = make_dpr_batch()
    loss = dpr_loss(dpr, batch, use_hard_negatives=True)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(dpr.parameters(), 1.0)
    optimizer.step()
    if step % 50 == 0:
        print(f"  step {step:3d}: loss={loss.item():.4f}")

In [ ]:
# ---- Retrieval Pipeline ----
# After training: encode all passages ONCE (offline), store in FAISS.
# At query time: encode question, search FAISS, return top-k passages.

class DPRRetriever:
    """
    Offline: index all passages.
    Online: encode query, search index.
    """
    def __init__(self, model: DPRModel):
        self.model = model
        self.passage_embeddings = None
        self.passages = []
    
    @torch.no_grad()
    def index_passages(self, passage_tokens: List[torch.Tensor], 
                       passage_texts: List[str], batch_size=32):
        """Encode all passages and build flat index."""
        self.model.eval()
        self.passages = passage_texts
        all_embs = []
        for i in range(0, len(passage_tokens), batch_size):
            batch = torch.stack(passage_tokens[i:i+batch_size])
            emb = self.model.encode_passage(batch)
            all_embs.append(emb.cpu().numpy())
        self.passage_embeddings = np.vstack(all_embs)  # [N, d]
        print(f"Indexed {len(self.passages)} passages, shape: {self.passage_embeddings.shape}")
    
    @torch.no_grad()
    def retrieve(self, question_tokens: torch.Tensor, k=5):
        """Encode question, find top-k passages."""
        self.model.eval()
        q_emb = self.model.encode_question(question_tokens.unsqueeze(0))
        q_emb = q_emb.cpu().numpy()  # [1, d]
        
        # Dot product with all passage embeddings
        scores = (self.passage_embeddings @ q_emb.T).flatten()
        top_k = np.argsort(-scores)[:k]
        return [(self.passages[i], float(scores[i])) for i in top_k]


# Build a small passage corpus
N_PASSAGES = 100
T_P = 20
passage_tokens_list = [torch.randint(1, VOCAB, (T_P,)) for _ in range(N_PASSAGES)]
passage_texts = [f"Passage about topic {i}" for i in range(N_PASSAGES)]

retriever = DPRRetriever(dpr)
retriever.index_passages(passage_tokens_list, passage_texts)

# Query
query = torch.randint(1, VOCAB, (10,))
results = retriever.retrieve(query, k=3)
print("\nTop-3 retrieved passages:")
for text, score in results:
    print(f"  {text}  (score={score:.4f})")

In [ ]:
# ---- DPR vs BM25: When to use which ----

print("=== DPR vs BM25 ===")
print()
print("BM25 wins when:")
print("  - Query contains rare/specialized terms (drug names, error codes)")
print("  - Domain is narrow and vocabulary is controlled")
print("  - No labeled data for training a dense retriever")
print("  - Need interpretability (which words caused the match?)")
print()
print("DPR wins when:")
print("  - Query and passage use different words for the same concept")
print("  - Synonymy, paraphrase, or language variability is common")
print("  - Sufficient labeled (question, passage) pairs available")
print()
print("Hybrid wins most of the time:")
print("  - Run BM25 + DPR in parallel, merge results with Reciprocal Rank Fusion (RRF)")
print("  - Or: use BM25 to get top-1000, then re-rank with a cross-encoder")
print()
print("Reciprocal Rank Fusion (RRF):")
print("  RRF_score(d) = sum_over_systems( 1 / (k + rank_in_system(d)) )")
print("  k=60 is standard. Robust to different score scales.")

def reciprocal_rank_fusion(rankings: list[list], k=60):
    """Fuse multiple ranked lists by Reciprocal Rank Fusion."""
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])

bm25_ranking = [3, 7, 1, 12, 5, 9]
dpr_ranking  = [7, 3, 15, 1, 8, 2]
fused = reciprocal_rank_fusion([bm25_ranking, dpr_ranking])
print("\nRRF example:")
print(f"  BM25: {bm25_ranking}")
print(f"  DPR:  {dpr_ranking}")
print(f"  RRF:  {[doc for doc, _ in fused[:6]]}")

## Summary

**DPR = bi-encoder trained with in-batch negative contrastive loss.**

**Pipeline:** Train encoders → encode all passages offline → build FAISS index → at query time: encode query, search FAISS in O(1) per cluster.

**Hard negatives matter:** BM25-retrieved but non-relevant passages are the most useful negatives — they look relevant (share keywords) but aren't. Training on these forces the model to learn true semantic relevance.

**Cross-encoder re-ranking:** A cross-encoder sees both query and passage together (full attention across them) and re-scores top-k DPR results. Much more accurate but too slow to run on all passages.

---
**Next:** 07.5 — RAG from scratch